# Patient-Specific Seizure Detection — LORO Evaluation

Fixed-hyperparameter pipeline (no SVM tuning). For each patient:
LORO over seizure records, per-fold operating-point calibration on
out-of-fold training scores, SzCORE event-level scoring.

**Set your hyperparameters once in the CONFIG cell.** AP / ROC-AUC are
reported per fold as leakage-free diagnostics only — they select nothing.

## Imports

In [ ]:
import os
# single-thread BLAS so the outer joblib parallelism over folds is clean
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import h5py
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

from timescoring import scoring
from timescoring.annotations import Annotation

## CONFIG

Everything you change between runs lives here. The SVM hyperparameters are
**fixed** — set them once; nothing is tuned on test data.

In [ ]:
# --- patients to run ---
PATIENTS = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24
# PATIENTS = ["chb01"]                              # or a single patient

# --- paths ---
SEIZURE_DIR     = Path("Features/Seizure")
NON_SEIZURE_DIR = Path("Features/Non_Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- FIXED SVM configs (pre-specified; reported transparently, NOT selected) ---
# Each is run in full over all patients. Add/remove rows here.
# Keep "name" short and unique — it tags every output file and row.
CONFIGS = [
    # softer boundary, more regularization
    {"name": "A", "C": 0.1, "gamma": "scale", "percentile": 5},
    {"name": "B", "C": 0.1,  "gamma": "scale", "percentile": 10},
    {"name": "C", "C": 0.1,  "gamma": "scale", "percentile": 15},
    {"name": "D", "C": 0.1,  "gamma": 0.01, "percentile": 5},
    {"name": "E", "C": 0.1,  "gamma": 0.01,   "percentile": 10},
    {"name": "F", "C": 0.1,  "gamma": 0.01,   "percentile": 15},
]
# --- windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- operating-point calibration (threshold + persistence k) ---
THRESH_GRID       = None                       # None -> per-fold score quantiles
K_GRID            = [1, 2, 3, 4, 6, 8]         # consecutive-window persistence
FP_PER_DAY_BUDGET = 5                           # a-priori FP/day ceiling

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# --- parallelism over LORO folds ---
OUTER_N_JOBS = 8

# --- outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # one combined summary, tagged by config

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


def make_pipe(cfg):
    # Memory caching removed on purpose (it wrote a growing 'cache/' folder
    # to disk and gave no speedup). cache_size below is the in-RAM libsvm
    # kernel buffer, per worker — not disk.
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight="balanced", cache_size=500)),
    ])

## Load one patient

Returns `seizure_records` and `non_records` dicts: `{record_name: (X, y)}`.

In [ ]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Event-scoring & calibration helpers

In [ ]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask


def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def calibrate(oof_record_scores):
    # Selects threshold + k (the OPERATING POINT) on out-of-fold train scores.
    # This is not model selection — the SVM is already fixed.
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

## LORO for one patient

Fixed SVM. Per fold: fit on all-but-one seizure record, get OOF train scores
to calibrate (threshold, k), score the held-out record. Seizure-free records
get one global model + global (threshold, k) so their false alarms count.

`ap` / `roc_auc` are computed per fold on the held-out record as **diagnostics
only** — they are printed, never used to choose anything.

In [ ]:
def build_train(seizure_records, exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)


def oof_train_record_scores(seizure_records, exclude, cfg):
    Xtr, ytr, grps = build_train(seizure_records, exclude=exclude)
    oof = cross_val_predict(make_pipe(cfg), Xtr, ytr, groups=grps,
                            cv=LeaveOneGroupOut(),
                            method="decision_function", n_jobs=1)
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        out.append((name, oof[m], ytr[m], duration_of(ytr[m])))
    return out


def diag_scores(y, sc):
    # leakage-free per-fold ranking diagnostics; select nothing
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)


def process_seizure_fold(seizure_records, test_record, cfg):
    Xtr, ytr, grps = build_train(seizure_records, exclude=test_record)
    model = make_pipe(cfg).fit(Xtr, ytr)
    oof = oof_train_record_scores(seizure_records, exclude=test_record, cfg=cfg)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap)

## Run one patient (build tables + return summary row)

In [ ]:
def run_patient(patient, cfg):
    seizure_records, non_records = load_patient(patient)
    if not seizure_records:
        print(f"[{cfg['name']}|{patient}] no seizure records — skipping")
        return None, None
    print(f"[{cfg['name']}|{patient}] seizure={len(seizure_records)} "
          f"non-seizure={len(non_records)}")

    # --- LORO over seizure folds (parallel) ---
    results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(seizure_records, r, cfg)
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        meta[res["record"]] = dict(type="seizure", roc=res["roc"], ap=res["ap"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- one global model + global (thr,k) for all seizure-free records ---
    if non_records:
        Xall, yall, gall = build_train(seizure_records, exclude=None)
        full_model = make_pipe(cfg).fit(Xall, yall)
        oof_full = oof_train_record_scores(seizure_records, exclude=None, cfg=cfg)
        thr_fp, k_fp = calibrate(oof_full)
        print(f"  [NON-MODEL] thr={thr_fp:.3f} k={k_fp}")
        for name, (X, y) in non_records.items():
            sc = full_model.decision_function(X)
            test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
            meta[name] = dict(type="nonseizure", roc=float("nan"),
                              ap=float("nan"))

    # --- per-record pass: segment table + event totals ---
    TP = FP = FN = 0
    all_latencies, segment_rows = [], []
    for record, sc, y, dur, thr, k in test_outputs:
        is_seiz = meta[record]["type"] == "seizure"
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        if is_seiz:
            all_latencies.extend(detection_latencies(record, sc, thr, k))

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        if not is_seiz:
            acc = spec = sens = prec = f1 = np.nan
        segment_rows.append({
            "config": cfg["name"], "C": cfg["C"], "gamma": cfg["gamma"],
            "percentile": cfg["percentile"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")

    summary_row = {
        "config": cfg["name"], "C": cfg["C"], "gamma": cfg["gamma"],
        "percentile": cfg["percentile"],
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    return summary_row, segment_rows

## Run all patients

Writes one `SEGMENT_PER_RECORD_<patient>.csv` per patient and appends a row to
`RESULTS_SUMMARY.csv`. Re-runs overwrite the summary cleanly (no stale rows).

In [ ]:
all_summaries, all_segments = [], []

for cfg in CONFIGS:
    print(f"\n{'='*60}\nCONFIG {cfg['name']}: "
          f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
          f"\n{'='*60}")
    for patient in PATIENTS:
        try:
            srow, segrows = run_patient(patient, cfg)
            if srow is not None:
                all_summaries.append(srow)
                all_segments.extend(segrows)
        except Exception as e:
            print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
            continue

        # --- incremental save after EVERY patient, not just at the end ---
        # protects against crashes / timeouts / disconnects mid-run.
        # overwrites the same two files each time -> always reflects
        # everything completed so far.
        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv("SEGMENT_PER_RECORD_ALL.csv", index=False)

# final write (redundant with the last incremental save, kept for clarity)
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv("SEGMENT_PER_RECORD_ALL.csv", index=False)

print(f"\nWrote {SUMMARY_CSV} "
      f"({summary_df['config'].nunique()} configs x "
      f"{summary_df['patient'].nunique()} patients)")

# quick pooled view per config (event-level), for transparent reporting
if len(summary_df):
    pooled = (summary_df.groupby("config")
              .apply(lambda d: pd.Series({
                  "TP": d.TP.sum(), "FP": d.FP.sum(), "FN": d.FN.sum(),
                  "sensitivity": d.TP.sum()/(d.TP.sum()+d.FN.sum())
                                 if (d.TP.sum()+d.FN.sum()) else float("nan"),
                  "precision": d.TP.sum()/(d.TP.sum()+d.FP.sum())
                               if (d.TP.sum()+d.FP.sum()) else float("nan"),
                  "FP/day": d.FP.sum()/(d.total_length_hours.sum()/24)
                            if d.total_length_hours.sum() else float("nan"),
              }), include_groups=False)
              .reset_index())
    print("\nPooled event-level by config:")
    print(pooled.to_string(index=False))
summary_df